# 03 · Extracción a tablas, sin corrección

## Objetivo
Escribir en `data_procesados/` la estadística y las estaciones del survey, y registrar qué transformación se aplicó.

## Alcance
La extracción no borra ni sustituye celdas del Excel. No hay filtro de outliers ni conversión silenciosa de BS&W.


In [1]:
from pathlib import Path
import os, sys
aqui = Path.cwd().resolve()
expediente = next(c for c in [aqui, *aqui.parents] if (c / "dataset.md").is_file() and (c / "entrada").is_dir())
os.chdir(expediente)
if str(expediente / "src") not in sys.path:
    sys.path.insert(0, str(expediente / "src"))
libro = expediente / "entrada" / "BES_jun.xlsx"

from pathlib import Path
from ingesta.leer_bes_jun import leer_libro
from ingesta.survey import leer_survey
from analisis.estadisticas import guardar_tablas
hojas = leer_libro(libro)
antes = libro.stat().st_size
rutas = guardar_tablas(hojas, leer_survey(libro))
for nombre, ruta in rutas.items():
    print(nombre, ruta.relative_to(expediente), "bytes", ruta.stat().st_size)
print(Path(rutas["nota"]).read_text())
print("bytes_excel_antes", antes, "bytes_excel_despues", libro.stat().st_size)
assert antes == libro.stat().st_size


operacion data_procesados/estadisticas_operacion.csv bytes 4842
survey data_procesados/estadisticas_survey.csv bytes 4321
estaciones data_procesados/survey_estaciones.csv bytes 13336
nota data_procesados/transformaciones.md bytes 340
# Transformaciones

No se borró, corrigió ni imputó ningún valor del Excel.
La copia de trabajo no se guardó desde openpyxl.
`survey_estaciones.csv` copia las filas numéricas del bloque `Measured`.
Se omitió la fila de unidades que sigue al encabezado (`Depth (ft)`).
Cada hoja queda identificada por su nombre. No se unieron pozos.

bytes_excel_antes 329308 bytes_excel_despues 329308


## Transformación ejecutada
`guardar_tablas` escribe cuatro archivos. `estadisticas_operacion.csv` tiene una fila por hoja y por variable de operación. `estadisticas_survey.csv` resume el bloque Measured de cada hoja. `survey_estaciones.csv` copia las estaciones con profundidad medida numérica. `transformaciones.md` dice que no se imputó y que se omitió la fila de unidades bajo el encabezado.

La celda imprime el tamaño de cada archivo derivado y comprueba que `entrada/BES_jun.xlsx` conserva el mismo tamaño en bytes antes y después. Ese control no reemplaza al SHA-256 del cuaderno 00; solo muestra que esta extracción no reescribió el libro.

No se construyó un esquema común con otros Excel.
